In [ ]:
import os
GROQ_API_KEY="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"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [ ]:
!pip install langchain langchain-groq langchain-community \
            pdfplumber python-docx pydantic groq docx2txt

In [ ]:
"""Pydantic schema for structured resume extraction output."""

from pydantic import BaseModel, Field
from typing import List, Optional

class EducationEntry(BaseModel):
    degree: str = Field(description="Degree title e.g. BSc Computer Science")
    institution: str = Field(description="University or school name")
    year: Optional[str] = Field(description="Graduation year or date range")
    gpa: Optional[str] = Field(description="GPA if mentioned")

class ExperienceEntry(BaseModel):
    title: str = Field(description="Job title or role")
    company: str = Field(description="Company or organization name")
    duration: Optional[str] = Field(description="Date range e.g. Jun 2022 – Aug 2023")
    bullets: List[str] = Field(description="Achievement bullet points")

class ProjectEntry(BaseModel):
    name: str
    description: str
    tech_stack: List[str]

class ResumeData(BaseModel):
    full_name: Optional[str]
    email: Optional[str]
    skills: List[str] = Field(description="All technical skills, lowercase, deduplicated")
    education: List[EducationEntry]
    experience: List[ExperienceEntry]
    projects: List[ProjectEntry]
    courses_and_certifications: List[str]
    languages: List[str] = Field(description="Spoken languages if mentioned")
    summary: Optional[str] = Field(description="Objective or summary section")

In [ ]:
"""
Resume extraction pipeline using LangChain + Groq (free, fast).
Replaces model2.py + extractor.py entirely.
Supports PDF, DOCX, TXT input.
"""

import time
from pathlib import Path
from typing import Union

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# The ResumeData class is defined in an earlier cell and is globally available.
# from resume_schema import ResumeData

# ── LLM setup ────────────────────────────────────────────────────────────────

def get_llm():
    """Return Groq LLM. Falls back to Ollama if GROQ_API_KEY not set."""
    import os
    if os.getenv("GROQ_API_KEY"):
        return ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0,           # deterministic output
            max_tokens=2048,
        )
    else:
        # local fallback — run: ollama pull llama3
        from langchain_ollama import ChatOllama
        return ChatOllama(model="llama3", temperature=0)

# ── Document loading ──────────────────────────────────────────────────────────

def load_document(file_path: Union[str, Path]) -> str:
    """
    Load text from PDF, DOCX, or TXT.
    Returns raw text string.
    Raises ValueError for unsupported types.
    """
    path = Path(file_path)
    suffix = path.suffix.lower()

    loaders = {
        ".pdf":  PyPDFLoader,
        ".docx": Docx2txtLoader,
        ".txt":  TextLoader,
    }

    if suffix not in loaders:
        raise ValueError(f"Unsupported file type: {suffix}. Use PDF, DOCX, or TXT.")

    loader = loaders[suffix](str(path))
    docs = loader.load()

    if not docs or not any(d.page_content.strip() for d in docs):
        raise RuntimeError("File parsed but produced no text. May be a scanned image PDF.")

    return "\n\n".join(d.page_content for d in docs)

# ── Chunking for long resumes ─────────────────────────────────────────────────

def chunk_if_needed(text: str, max_chars: int = 6000) -> str:
    """
    Most resumes fit in one LLM call.
    If over limit, take the most information-dense first chunk.
    For very long CVs, you could map-reduce — but 6000 chars
    covers 99% of real resumes comfortably.
    """
    if len(text) <= max_chars:
        return text

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chars,
        chunk_overlap=200,
        separators=["\n\n", "\n", " "]
    )
    chunks = splitter.split_text(text)
    return chunks[0]  # first chunk has header, skills, education — most useful

# ── Prompt ────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are an expert resume parser.\nExtract ALL structured information from the resume text provided.\nBe thorough — do not skip skills, even if they appear in project descriptions.\nNormalize all skills to lowercase (e.g. \"Python\", \"PYTHON\" → \"python\").\nRemove duplicates from skills list.\nIf a field is not present, return an empty list or null.\nReturn ONLY valid JSON matching the schema. No explanation, no markdown fences."""

# ── Main extraction function ──────────────────────────────────────────────────

def extract_resume(
    file_path: Union[str, Path, None] = None,
    raw_text: Union[str, None] = None,
) -> dict:
    """
    Main entry point. Pass either a file path or raw text string.
    Returns a dict ready for your API response and embedding layer.
    """
    start = time.time()

    # 1. Load text
    if file_path:
        text = load_document(file_path)
    elif raw_text:
        text = raw_text.strip()
    else:
        raise ValueError("Provide either file_path or raw_text.")

    # 2. Chunk if needed
    text = chunk_if_needed(text)

    # 3. Parser + prompt
    parser = PydanticOutputParser(pydantic_object=ResumeData)

    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "Extract from this resume:\n\n{resume_text}\n\n{format_instructions}"),
    ])

    # 4. Chain
    llm = get_llm()
    chain = prompt | llm | parser

    result: ResumeData = chain.invoke({
        "resume_text": text,
        "format_instructions": parser.get_format_instructions(),
    })

    elapsed_ms = int((time.time() - start) * 1000)

    # 5. Return clean dict for API + embedding layer
    return {
        "success": True,
        "extractedData": {
            "fullName": result.full_name,
            "email": result.email,
            "summary": result.summary,
            "cleanSkills": result.skills,             # ← ready for embedding
            "education": [e.model_dump() for e in result.education],
            "experience": [e.model_dump() for e in result.experience],
            "projects": [e.model_dump() for e in result.projects],
            "coursesAndCertifications": result.courses_and_certifications,
            "languages": result.languages,
        },
        "meta": {
            "processingTimeMs": elapsed_ms,
            "charsProcessed": len(text),
        }
    }

## Resume Extraction

First, please upload your resume file (PDF, DOCX, or TXT) using the file uploader below.

In [ ]:
from google.colab import files

uploaded = files.upload()

# Get the filename of the uploaded file
if uploaded:
    uploaded_filename = list(uploaded.keys())[0]
    print(f"Uploaded file: {uploaded_filename}")
else:
    print("No file uploaded.")

Saving Rodina Mohamed-Resume v.2 (1).docx to Rodina Mohamed-Resume v.2 (1).docx
Uploaded file: Rodina Mohamed-Resume v.2 (1).docx


Now, let's extract the information from the uploaded resume. Ensure the `uploaded_filename` variable holds the name of the file you want to process.

In [ ]:
try:
    if 'uploaded_filename' in locals() and uploaded_filename:
        # Check if extract_resume is defined before calling
        if 'extract_resume' not in globals() or not callable(extract_resume):
            raise RuntimeError("The 'extract_resume' function is not defined. Please ensure all preceding code cells (especially those defining the resume schema and extraction logic) have been executed.")
        extracted_data = extract_resume(file_path=uploaded_filename)
        import json
        print(json.dumps(extracted_data, indent=2))
    else:
        print("No resume file detected. Please upload a file first.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")

{
  "success": true,
  "extractedData": {
    "fullName": "Rodina Mohamed Saeed",
    "email": "rodinamo2003@gmail.com",
    "summary": "Applied AI Engineer and Full-Stack Developer with a specialized focus on Agentic AI and production-grade LLM implementation.",
    "cleanSkills": [
      "genai",
      "llms",
      "sft",
      "rag",
      "langchain",
      "agentic workflows",
      "prompt engineering",
      "ml",
      "ai",
      "rl",
      "neural networks",
      "scikit-learn",
      "backend",
      "python",
      "fastapi",
      "rest apis",
      "java",
      "oop",
      "multithreading",
      "web",
      "mern",
      "vue.js",
      "javascript",
      "html/css",
      "data",
      "mysql",
      "mongodb",
      "chroma",
      "pandas",
      "numpy",
      "tools",
      "docker",
      "git",
      "ci/cd",
      "technical mentorship",
      "ai system design",
      "trade-off analysis"
    ],
    "education": [
      {
        "degree": "Bachelor\u2019